# PRIS116 Data Analysis - Danish Price Indices

This notebook demonstrates how to fetch and analyze data from the PRIS116 table using Statistics Denmark's API. PRIS116 typically contains price indices and inflation-related data.

## Table of Contents
1. Setup and Import Libraries
2. Initialize DST API Connection
3. Explore Table Structure
4. Fetch Data with Custom Parameters
5. Data Analysis and Visualization
6. Time Series Analysis

## 1. Setup and Import Libraries

In [ ]:
# Import required libraries
import sys
import os

# Add the current directory to Python path to import dstapi
sys.path.append(os.getcwd())

from dstapi import DstApi
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import requests
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully!")
print(f"Current working directory: {os.getcwd()}")

## 2. Initialize DST API Connection

In [ ]:
# Initialize the DST API for PRIS116 table
dst_api = DstApi('PRIS116')

print("DST API initialized for table PRIS116")
print(f"API endpoint: {dst_api.apiip}")
print(f"Table name: {dst_api.tablename}")

## 3. Explore Table Structure

In [ ]:
# Get table summary
print("=== PRIS116 Table Summary ===")
table_summary = dst_api.tablesummary(verbose=True, language="en")
print("\nTable structure:")
display(table_summary)

In [ ]:
# Explore available variables in the table
print("=== Available Variables ===")
for idx, row in table_summary.iterrows():
    var_name = row['variable name']
    num_values = row['# values']
    first_val = row['First value']
    last_val = row['Last value']
    is_time = row['Time variable']
    
    print(f"\nVariable: {var_name}")
    print(f"  Number of values: {num_values}")
    print(f"  Range: {first_val} to {last_val}")
    print(f"  Time variable: {is_time}")
    
    # Show first few values for each variable
    try:
        var_levels = dst_api.variable_levels(var_name, language="en")
        print(f"  First 5 values:")
        display(var_levels.head())
    except Exception as e:
        print(f"  Could not retrieve variable levels: {e}")

## 4. Fetch Data with Custom Parameters

Based on your example, we'll create specific parameter queries for PRIS116 data.

In [ ]:
# First, let's try the direct API approach similar to your example
print("=== Direct API Call Example ===")

# Example parameters for PRIS116 (adjust based on actual table structure)
params_direct = {
    'table': 'PRIS116',
    'format': 'BULK',
    'lang': 'en',
    'variables': [
        {'code': 'Tid', 'values': ['*']}  # Get all time periods
    ]
}

try:
    r = requests.post('https://api.statbank.dk/v1/data', json=params_direct)
    print(f"Response status: {r.status_code}")
    print(f"First 500 characters of response:")
    print(r.text[:500])
    print("...")
except Exception as e:
    print(f"Direct API call failed: {e}")

In [ ]:
# Create more specific parameters based on table structure
# Let's start with a base parameter set and then customize

print("=== Creating Base Parameters ===")
base_params = dst_api.define_base_params(language="en")
print("Base parameters structure:")
print(f"Table: {base_params['table']}")
print(f"Format: {base_params['format']}")
print(f"Language: {base_params['lang']}")
print(f"Number of variables: {len(base_params['variables'])}")

print("\nVariables in base parameters:")
for var in base_params['variables']:
    print(f"  {var['code']}: {var['values']}")

In [ ]:
# Create custom parameters for specific data extraction
# This will be adjusted based on what we discover about the table structure

print("=== Fetching Sample Data ===")

# Start with a limited query to avoid downloading too much data
try:
    # Get data with limited time range (last 5 years)
    current_year = datetime.now().year
    recent_years = [str(year) for year in range(current_year-5, current_year+1)]
    
    # Create custom parameters (will need to adjust based on actual table structure)
    custom_params = base_params.copy()
    
    # Limit time variable to recent years if 'Tid' exists
    for var in custom_params['variables']:
        if var['code'].lower() in ['tid', 'time']:
            # Try to limit to recent years
            var['values'] = ['*']  # Start with all, then we'll see what's available
            break
    
    print("Attempting to fetch data with custom parameters...")
    
    # Use override_warning=True for this exploratory fetch
    sample_data = dst_api.get_data(params=custom_params, language="en", override_warning=True)
    
    if sample_data is not None and not sample_data.empty:
        print(f"Successfully fetched data!")
        print(f"Data shape: {sample_data.shape}")
        print(f"Columns: {list(sample_data.columns)}")
        print("\nFirst few rows:")
        display(sample_data.head(10))
        
        print("\nData types:")
        print(sample_data.dtypes)
        
        print("\nBasic statistics:")
        display(sample_data.describe())
        
    else:
        print("No data retrieved or data is empty")
        
except Exception as e:
    print(f"Error fetching data: {e}")
    print("Let's try a different approach...")

In [ ]:
# Alternative approach: Try to fetch data with minimal parameters
print("=== Alternative Data Fetch Approach ===")

try:
    # Try with just time variable
    minimal_params = {
        'table': 'PRIS116',
        'format': 'BULK',
        'lang': 'en',
        'variables': [
            {'code': 'TID', 'values': ['2020M01', '2020M02', '2020M03']},  # Try specific months
        ]
    }
    
    print("Trying minimal parameters...")
    r = requests.post('https://api.statbank.dk/v1/data', json=minimal_params)
    
    if r.status_code == 200:
        print("Success! Parsing data...")
        from io import StringIO
        df = pd.read_csv(StringIO(r.text), sep=";", decimal=",")
        print(f"Data shape: {df.shape}")
        print(f"Columns: {list(df.columns)}")
        display(df.head())
    else:
        print(f"Request failed with status {r.status_code}")
        print(f"Response: {r.text[:500]}")
        
except Exception as e:
    print(f"Alternative approach failed: {e}")

## 5. Data Analysis and Visualization

Once we have successfully fetched the data, we'll analyze it.

In [ ]:
# This cell will be populated once we successfully fetch data
# For now, let's create a mock analysis structure

print("=== Data Analysis Section ===")
print("This section will be populated once we successfully fetch PRIS116 data")

# If we have data in the variable 'sample_data' or 'df', we can analyze it
try:
    # Check if we have any data from previous cells
    if 'sample_data' in locals() and sample_data is not None and not sample_data.empty:
        data_to_analyze = sample_data
        print("Using sample_data for analysis")
    elif 'df' in locals() and df is not None and not df.empty:
        data_to_analyze = df
        print("Using df for analysis")
    else:
        print("No data available for analysis yet")
        data_to_analyze = None
        
    if data_to_analyze is not None:
        print(f"\nAnalyzing data with shape: {data_to_analyze.shape}")
        
        # Basic data exploration
        print("\n=== Data Overview ===")
        print(f"Columns: {list(data_to_analyze.columns)}")
        print(f"Data types:")
        print(data_to_analyze.dtypes)
        
        # Look for time columns
        time_cols = [col for col in data_to_analyze.columns if 'tid' in col.lower() or 'time' in col.lower() or 'år' in col.lower()]
        print(f"\nPotential time columns: {time_cols}")
        
        # Look for value columns
        numeric_cols = data_to_analyze.select_dtypes(include=[np.number]).columns.tolist()
        print(f"Numeric columns: {numeric_cols}")
        
        # Show unique values for categorical columns
        categorical_cols = data_to_analyze.select_dtypes(include=['object']).columns.tolist()
        print(f"\nCategorical columns: {categorical_cols}")
        
        for col in categorical_cols[:3]:  # Show first 3 categorical columns
            unique_vals = data_to_analyze[col].unique()
            print(f"\n{col} unique values ({len(unique_vals)}):")
            print(unique_vals[:10])  # Show first 10 unique values
            if len(unique_vals) > 10:
                print("...")
                
except Exception as e:
    print(f"Error in data analysis: {e}")

In [ ]:
# Visualization section
print("=== Data Visualization ===")

try:
    if 'data_to_analyze' in locals() and data_to_analyze is not None:
        # Create visualizations based on the data structure
        
        # 1. Basic distribution plots for numeric columns
        numeric_cols = data_to_analyze.select_dtypes(include=[np.number]).columns.tolist()
        
        if numeric_cols:
            print(f"Creating distribution plots for numeric columns: {numeric_cols}")
            
            fig, axes = plt.subplots(len(numeric_cols), 1, figsize=(12, 4*len(numeric_cols)))
            if len(numeric_cols) == 1:
                axes = [axes]
                
            for i, col in enumerate(numeric_cols):
                axes[i].hist(data_to_analyze[col].dropna(), bins=30, alpha=0.7)
                axes[i].set_title(f'Distribution of {col}')
                axes[i].set_xlabel(col)
                axes[i].set_ylabel('Frequency')
                
            plt.tight_layout()
            plt.show()
        
        # 2. Time series plot if we can identify time and value columns
        time_cols = [col for col in data_to_analyze.columns if 'tid' in col.lower() or 'time' in col.lower()]
        
        if time_cols and numeric_cols:
            print(f"\nCreating time series plots...")
            time_col = time_cols[0]
            
            # Try to convert time column to datetime
            try:
                data_to_analyze[time_col] = pd.to_datetime(data_to_analyze[time_col])
                
                for value_col in numeric_cols[:2]:  # Plot first 2 numeric columns
                    plt.figure(figsize=(12, 6))
                    plt.plot(data_to_analyze[time_col], data_to_analyze[value_col], marker='o', linewidth=2, markersize=4)
                    plt.title(f'{value_col} over Time')
                    plt.xlabel('Time')
                    plt.ylabel(value_col)
                    plt.xticks(rotation=45)
                    plt.grid(True, alpha=0.3)
                    plt.tight_layout()
                    plt.show()
                    
            except Exception as e:
                print(f"Could not create time series plot: {e}")
        
        print("\nVisualization complete!")
        
    else:
        print("No data available for visualization")
        
except Exception as e:
    print(f"Error in visualization: {e}")

## 6. Interactive Plotly Visualizations

In [ ]:
# Create interactive visualizations with Plotly
print("=== Interactive Plotly Visualizations ===")

try:
    if 'data_to_analyze' in locals() and data_to_analyze is not None:
        
        # 1. Interactive time series plot
        time_cols = [col for col in data_to_analyze.columns if 'tid' in col.lower() or 'time' in col.lower()]
        numeric_cols = data_to_analyze.select_dtypes(include=[np.number]).columns.tolist()
        
        if time_cols and numeric_cols:
            time_col = time_cols[0]
            
            fig = make_subplots(
                rows=len(numeric_cols), cols=1,
                subplot_titles=[f'{col} over Time' for col in numeric_cols],
                shared_xaxes=True
            )
            
            for i, col in enumerate(numeric_cols, 1):
                fig.add_trace(
                    go.Scatter(
                        x=data_to_analyze[time_col],
                        y=data_to_analyze[col],
                        mode='lines+markers',
                        name=col,
                        line=dict(width=2),
                        marker=dict(size=4)
                    ),
                    row=i, col=1
                )
            
            fig.update_layout(
                title='PRIS116 Price Indices Over Time',
                height=300 * len(numeric_cols),
                showlegend=True
            )
            
            fig.update_xaxes(title_text="Time", row=len(numeric_cols), col=1)
            
            fig.show()
        
        # 2. Correlation heatmap if we have multiple numeric columns
        if len(numeric_cols) > 1:
            corr_matrix = data_to_analyze[numeric_cols].corr()
            
            fig_heatmap = px.imshow(
                corr_matrix,
                color_continuous_scale='RdBu',
                color_continuous_midpoint=0,
                title="Correlation Matrix of Price Indices",
                width=600,
                height=600
            )
            
            fig_heatmap.show()
        
        # 3. Box plots for categorical variables vs numeric values
        categorical_cols = data_to_analyze.select_dtypes(include=['object']).columns.tolist()
        
        if categorical_cols and numeric_cols:
            cat_col = categorical_cols[0]
            num_col = numeric_cols[0]
            
            # Only create box plot if categorical column has reasonable number of categories
            if data_to_analyze[cat_col].nunique() <= 20:
                fig_box = px.box(
                    data_to_analyze,
                    x=cat_col,
                    y=num_col,
                    title=f'{num_col} by {cat_col}'
                )
                
                fig_box.update_xaxes(tickangle=45)
                fig_box.show()
        
        print("Interactive visualizations complete!")
        
    else:
        print("No data available for interactive visualization")
        # Create a sample visualization to show the structure
        print("Creating sample visualization structure...")
        
        # Sample data for demonstration
        sample_dates = pd.date_range('2020-01-01', '2023-12-01', freq='M')
        sample_values = np.random.randn(len(sample_dates)).cumsum() + 100
        
        fig_sample = go.Figure()
        fig_sample.add_trace(go.Scatter(
            x=sample_dates,
            y=sample_values,
            mode='lines+markers',
            name='Sample Price Index',
            line=dict(width=3),
            marker=dict(size=6)
        ))
        
        fig_sample.update_layout(
            title='Sample PRIS116 Visualization Structure',
            xaxis_title='Time',
            yaxis_title='Price Index',
            width=800,
            height=400
        )
        
        fig_sample.show()
        
except Exception as e:
    print(f"Error in interactive visualization: {e}")

## 7. Advanced Analysis Functions

In [ ]:
# Define helper functions for PRIS116 analysis

def calculate_inflation_rate(price_index, periods=12):
    """
    Calculate inflation rate from price index data.
    
    Parameters:
    price_index: pandas Series with price index values
    periods: number of periods for year-over-year calculation (default 12 for monthly data)
    
    Returns:
    pandas Series with inflation rates
    """
    return price_index.pct_change(periods=periods) * 100

def analyze_price_trends(data, time_col, price_col):
    """
    Analyze price trends and provide summary statistics.
    """
    if time_col not in data.columns or price_col not in data.columns:
        return None
    
    # Sort by time
    data_sorted = data.sort_values(time_col)
    
    # Calculate basic statistics
    stats = {
        'mean': data_sorted[price_col].mean(),
        'std': data_sorted[price_col].std(),
        'min': data_sorted[price_col].min(),
        'max': data_sorted[price_col].max(),
        'first_value': data_sorted[price_col].iloc[0],
        'last_value': data_sorted[price_col].iloc[-1],
        'total_change': ((data_sorted[price_col].iloc[-1] / data_sorted[price_col].iloc[0]) - 1) * 100
    }
    
    return stats

def create_price_summary_report(data):
    """
    Create a comprehensive summary report of price data.
    """
    print("=== PRIS116 Data Summary Report ===")
    print(f"Data period: {data.shape[0]} observations")
    print(f"Variables: {data.shape[1]} columns")
    
    # Identify time and numeric columns
    time_cols = [col for col in data.columns if 'tid' in col.lower() or 'time' in col.lower()]
    numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()
    
    if time_cols:
        print(f"\nTime column: {time_cols[0]}")
        print(f"Time range: {data[time_cols[0]].min()} to {data[time_cols[0]].max()}")
    
    if numeric_cols:
        print(f"\nPrice/Index columns: {numeric_cols}")
        
        for col in numeric_cols:
            if time_cols:
                trends = analyze_price_trends(data, time_cols[0], col)
                if trends:
                    print(f"\n{col} Analysis:")
                    print(f"  Mean: {trends['mean']:.2f}")
                    print(f"  Std Dev: {trends['std']:.2f}")
                    print(f"  Range: {trends['min']:.2f} - {trends['max']:.2f}")
                    print(f"  Total Change: {trends['total_change']:.2f}%")

# Apply analysis functions if we have data
try:
    if 'data_to_analyze' in locals() and data_to_analyze is not None:
        create_price_summary_report(data_to_analyze)
    else:
        print("Analysis functions defined. Will be applied once data is successfully fetched.")
except Exception as e:
    print(f"Error in advanced analysis: {e}")

## 8. Export Data and Results

In [ ]:
# Export data and analysis results
print("=== Data Export Section ===")

try:
    if 'data_to_analyze' in locals() and data_to_analyze is not None:
        # Export to CSV
        output_filename = f"pris116_data_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
        data_to_analyze.to_csv(output_filename, index=False)
        print(f"Data exported to: {output_filename}")
        
        # Export summary statistics
        summary_filename = f"pris116_summary_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
        with open(summary_filename, 'w') as f:
            f.write("PRIS116 Data Analysis Summary\n")
            f.write("=" * 40 + "\n")
            f.write(f"Analysis date: {datetime.now()}\n")
            f.write(f"Data shape: {data_to_analyze.shape}\n")
            f.write(f"Columns: {list(data_to_analyze.columns)}\n")
            f.write("\nData types:\n")
            f.write(str(data_to_analyze.dtypes))
            f.write("\n\nBasic statistics:\n")
            f.write(str(data_to_analyze.describe()))
        
        print(f"Summary exported to: {summary_filename}")
        
    else:
        print("No data available for export")
        
except Exception as e:
    print(f"Error in data export: {e}")

## Conclusion

This notebook provides a comprehensive framework for fetching and analyzing PRIS116 data from Statistics Denmark's API. The key components include:

1. **API Integration**: Uses the DstApi class to interact with Statistics Denmark's API
2. **Data Exploration**: Examines table structure and available variables
3. **Flexible Fetching**: Multiple approaches to retrieve data with custom parameters
4. **Analysis Functions**: Tools for calculating inflation rates and analyzing price trends
5. **Visualizations**: Both static (matplotlib) and interactive (Plotly) charts
6. **Export Capabilities**: Save data and results for further analysis

### Next Steps:
- Adjust parameters based on actual PRIS116 table structure
- Implement specific price index calculations
- Add time series forecasting models
- Create automated reporting functions
- Integrate with other economic indicators

### Notes:
- The actual table structure may require parameter adjustments
- Error handling is included for robust data fetching
- The notebook is designed to be modular and extensible
- All visualizations are responsive and interactive where possible